## Setting Up

In [1]:
# IMPORTS
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipyvolume as ipv
import neuropythy as ny

import torch
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SplineConv 

In [2]:
# CONFIGURE FREESURFER HOME
ny.config['freesurfer_home'] = '/opt/freesurfer'
ds = ny.config['freesurfer_home']

In [3]:
%matplotlib inline

## Basic Neuropythy Stuff

In [4]:
subject = ny.freesurfer_subject('fsaverage')

In [5]:
# SPLITTING INTO HEMISPHERES
subject_lh = subject.hemis['lh']
subject_rh = subject.hemis['rh']

In [6]:
# ASSIGNING SURFACE INFORMATION
lh_surface = subject_lh.surfaces['white']
rh_surface = subject_rh.surfaces['white']

In [7]:
# PLOT THE SURFACE
fig = ipv.figure(width=900, height=300)
ny.cortex_plot(lh_surface, color='thickness', cmap='cool', figure=fig)
ny.cortex_plot(rh_surface, color='thickness', cmap='cool', figure=fig)
ipv.show()

Container(figure=Figure(box_center=[0.5, 0.5, 0.5], box_size=[1.0, 1.0, 1.0], camera=PerspectiveCamera(fov=0.6…

In [8]:
# VERTICES OF THE HEMISPHERES
lh_vertices = lh_surface.coordinates
rh_vertices = rh_surface.coordinates

In [9]:
# MESHES OF THE HEMISPHERES
lh_faces = lh_surface.tess.faces
rh_faces = rh_surface.tess.faces

In [10]:
# EDGES OF THE HEMISPHERES
rh_edges = lh_surface.tess.edges
lh_edges = rh_surface.tess.edges

## Loading In Data

In [ ]:
# TODO When loading in multiple subjects, interpolate that information onto the fsaverage maps

In [11]:
# TODO Find a way to load both hemispheres
    # Currently, I'm thinking of loading in both hemisphere's edges and 
    # then transposing/reordering the right hemisphere's edges such that 
    # all of them come after the left hemisphere. Then, I'll probably 
    # have to add another column with a mask (or something similar) to 
    # differentiate between the left and right hemispheres.
if lh_edges.shape[0] == 2 and lh_edges.shape[1] != 2:
    lh_edges = lh_edges.T

if rh_edges.shape[0] == 2 and rh_edges.shape[1] != 2:
    rh_edges = rh_edges.T

# Create bidirectional edges
reversed_rh_edges = rh_edges[:, [1, 0]]
all_rh_edges = np.vstack([rh_edges, reversed_rh_edges])

# Build the feature object
edge_index = torch.tensor(all_rh_edges.T, dtype=torch.long)
x_coords = torch.tensor(rh_vertices.T, dtype=torch.float)
graph_data = Data(x=x_coords, edge_index=edge_index)

# Build the target
target_thickness = rh_surface.properties['thickness']
# TODO Is there an easier way to do this?
target_thickness = target_thickness.byteswap().view(target_thickness.dtype.newbyteorder())
graph_data.y = torch.tensor(target_thickness, dtype=torch.float).unsqueeze(1)

print(graph_data)

Data(x=[163842, 3], edge_index=[2, 983040], y=[163842, 1])


In [ ]:
# TODO Create dataset

In [ ]:
# TODO Split dataset into training, validation, and test

## Building Model

In [12]:
# TODO Build the actual model. This is just an example.
# TODO Change this to use SplineConv.
class ExampleGCN(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 16)
        self.conv2 = GCNConv(16, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        return x

In [13]:
model = ExampleGCN(in_channels=3, out_channels=1)

In [14]:
# Set up optimizer
lr = 0.01
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [15]:
# Set up loss
criterion = torch.nn.MSELoss()

In [16]:
# Set up training loop
model.train()

for epoch in range(200):
    optimizer.zero_grad()
    output = model(graph_data)
    loss = criterion(output, graph_data.y)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d} || MSE Loss: {loss.item():.4f}')

Epoch 000 || MSE Loss: 41.5442
Epoch 010 || MSE Loss: 6.6577
Epoch 020 || MSE Loss: 2.1108
Epoch 030 || MSE Loss: 0.9933
Epoch 040 || MSE Loss: 0.8667
Epoch 050 || MSE Loss: 0.5281
Epoch 060 || MSE Loss: 0.4445
Epoch 070 || MSE Loss: 0.3996
Epoch 080 || MSE Loss: 0.3794
Epoch 090 || MSE Loss: 0.3661
Epoch 100 || MSE Loss: 0.3549
Epoch 110 || MSE Loss: 0.3453
Epoch 120 || MSE Loss: 0.3370
Epoch 130 || MSE Loss: 0.3297
Epoch 140 || MSE Loss: 0.3232
Epoch 150 || MSE Loss: 0.3174
Epoch 160 || MSE Loss: 0.3121
Epoch 170 || MSE Loss: 0.3070
Epoch 180 || MSE Loss: 0.3023
Epoch 190 || MSE Loss: 0.2979


## Analysing Results

In [ ]:
# SANITY CHECKS
# print("GCN output shape:", output.shape)
# print("Output type:", type(output))
# print("Contains NaNs?:", torch.isnan(output).any().item())
# print("Min value:", output.min().item())
# print("Max value:", output.max().item())
# print("Mean value:", output.mean().item())

In [17]:
model.eval()
with torch.no_grad():
    predictions = model(graph_data).cpu().numpy().flatten()
ny.cortex_plot(rh_surface, color=predictions, cmap='cool')

Figure(box_center=[0.5, 0.5, 0.5], box_size=[1.0, 1.0, 1.0], camera=PerspectiveCamera(fov=0.644570721372708, p…